<a href="https://colab.research.google.com/github/Naylet92/Estudio_ambiental/blob/main/Quila_Plantilla.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MARCO GEOESPACIAL

## Área Natural Protegida: Sierra de Quila

**Tesista:** Naylet Hernández Sánchez  
**Director:** Viacheslav Shalisko  
**Doctorado:** Geografía y Ordenamiento Territorial  
**Proyecto:** "Evaluación comparativa de la pérdida de hábitat en territorios protegidos y no protegidos de Jalisco mediante percepción remota y aprendizaje automatizado en el periodo 2000-2020".

## Descripción del área

**1. Localización geográfica y contexto territorial**

El área se localiza en la región centro-occidental del estado de Jalisco, abarcando territorios de los municipios de Tecolotlán, Tenamaxtlán, Ameca y San Martín Hidalgo. Geográficamente, forma parte del sistema montañoso asociado al Eje Neovolcánico Transversal, lo que le confiere una elevada complejidad fisiográfica.

Asimismo, su altitud varía aproximadamente entre los 1,300 y 2,560 m s.n.m., lo que genera gradientes altitudinales significativos que influyen directamente en la distribución de la vegetación y en los procesos ecológicos.

**2. Características fisiográficas y geomorfológicas**

La Sierra de Quila presenta un relieve accidentado, dominado por sierras, mesetas disectadas, barrancas profundas y lomeríos. Esta configuración topográfica favorece la formación de microcuencas y una red hidrológica compleja.

En este sentido, la región constituye una zona importante de captación hídrica, alimentando diversos afluentes que forman parte de la cuenca del río Ameca. La interacción entre pendiente, litología y cobertura vegetal condiciona procesos de escorrentía, infiltración y erosión.

**3. Clima**

Predomina el tipo templado subhúmedo con lluvias en verano, aunque existen variaciones locales asociadas a la altitud. La temperatura media anual oscila entre 12 y 18 °C, mientras que la precipitación anual varía entre 800 y 1,200 mm.

De manera complementaria, la estacionalidad está bien definida, con una temporada de lluvias concentrada entre junio y octubre y un periodo seco el resto del año, lo que influye en la dinámica fenológica de los ecosistemas.

**4. Suelos y geología**

En términos edafológicos, predominan suelos como andosoles, luvisoles y regosoles, derivados principalmente de materiales volcánicos. Estos suelos presentan, en general, buena fertilidad natural, aunque su estabilidad está condicionada por la pendiente y el manejo del territorio.

Desde la perspectiva geológica, la Sierra de Quila forma parte de una provincia volcánica caracterizada por rocas ígneas extrusivas, lo cual influye en la morfología del paisaje y en la disponibilidad de nutrientes.

**5. Cobertura vegetal y uso del suelo**

En relación con la cobertura vegetal, el área presenta una notable diversidad de ecosistemas, destacando:

Bosques de pino (Pinus spp.)
Bosques de encino (Quercus spp.)
Bosques mixtos pino-encino
Parcheos de selva baja caducifolia en zonas de menor altitud

En este contexto, los bosques templados constituyen la cobertura dominante, asociados a condiciones climáticas más húmedas y frías en comparación con zonas costeras. Estos ecosistemas “son fundamentales para la regulación hidrológica y la conservación de la biodiversidad en México”.

No obstante, en zonas periféricas se identifican cambios de uso del suelo vinculados a actividades agrícolas, ganaderas y aprovechamientos forestales, lo que ha generado procesos de fragmentación.

**6. Biodiversidad y dinámica ecológica**

Desde una perspectiva ecológica, la Sierra de Quila alberga una alta diversidad biológica, incluyendo especies de flora y fauna de importancia ecológica y algunas bajo categoría de protección.

Además, los gradientes altitudinales y la heterogeneidad del paisaje favorecen la existencia de múltiples nichos ecológicos. En este sentido, la dinámica del área está determinada por la interacción entre factores climáticos, topográficos y antrópicos, lo que influye en procesos como la regeneración forestal, la sucesión ecológica y la resiliencia del ecosistema.

**7. Relevancia ambiental**

Finalmente, el área constituye un sistema estratégico para la conservación, debido a su papel en la recarga de acuíferos, la regulación climática local y la conectividad ecológica regional. En consecuencia, su estudio resulta fundamental para evaluar los efectos de las políticas de conservación y los cambios en el uso del suelo, especialmente en contextos de presión antrópica.

##Definir Variables

In [ ]:

#  DEFINICIÓN DE VARIABLES

# Prefijo del área
PREFIJO     = 'QLA'

# Nombre del área
NOMBRE      = 'Sierra_Quila'

RUTA_GPKG  = '/content/drive/MyDrive/ANP/AreadeProtecciondeFlorayFaunaSierradeQuila.gpkg'
LAYER_NAME  = None

# Carpeta de salida en Drive
RUTA_AOI    = '/content/drive/MyDrive/ANP/AOI'

# Rutas derivadas automáticamente (no tocar)
RUTA        = f'{RUTA_AOI}/aoi_{PREFIJO}.geojson'
RUTA_GEOJSON= RUTA
RUTA_CSV    = f'{RUTA_AOI}/{PREFIJO}_coordenadas.csv'

# CRS geográfico y UTM
CRS_GEO     = 4326
CRS_UTM     = 32613

# Proyecto de Google Earth Engine
GEE_PROJECT = 'ee-nayleths'

# Zoom inicial del mapa
ZOOM        = 12

# Estilo del área de estudio
STYLE_AREA  = {'color': 'blue', 'fillColor': '#0000ff30', 'weight': 1.5}

# Estilo del rectángulo rojo (bounding box)
STYLE_BBOX  = {'color': 'red', 'fillColor': '#00000000', 'weight': 2.5}

# Título del mapa
MAP_TITLE   = 'Área de estudio'

In [ ]:

import ee
import os
import geemap
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from google.colab import drive
from IPython.display import display, HTML

# Montar Google Drive
drive.mount('/content/drive')

# Autenticar e inicializar GEE
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

print("✓ Entorno listo")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Entorno listo


In [ ]:
# Crear carpeta AOI si no existe
os.makedirs(RUTA_AOI, exist_ok=True)

# Cargar y reproyectar a geográfico
gdf = gpd.read_file(RUTA_GPKG, layer=LAYER_NAME).to_crs(epsg=CRS_GEO)
minx, miny, maxx, maxy = gdf.total_bounds

# Reproyectar a UTM
gdf_utm = gdf.to_crs(epsg=CRS_UTM)
minx_u, miny_u, maxx_u, maxy_u = gdf_utm.total_bounds

# Guardar AOI como GeoJSON en Drive
gdf.to_file(RUTA_GEOJSON, driver='GeoJSON')
print(f"AOI guardado en: {RUTA_GEOJSON}")

# Guardar CSV
df = pd.DataFrame([{
    "Area"     : NOMBRE,
    "Lat_min"  : miny,
    "Lat_max"  : maxy,
    "Lon_min"  : minx,
    "Lon_max"  : maxx,
    "UTM_X_min": minx_u,
    "UTM_X_max": maxx_u,
    "UTM_Y_min": miny_u,
    "UTM_Y_max": maxy_u,
    "AOI"      : RUTA_GEOJSON
}])
df.to_csv(RUTA_CSV, index=False)
print(f"CSV guardado en: {RUTA_CSV}")

# Mostrar resultados

html = f"""
<div style="font-family: sans-serif; max-width: 420px;">

  <div style="border: 1px solid #ccc; border-radius: 8px; padding: 16px; margin-bottom: 12px;">
    <b style="font-size: 15px;">Sistema Geodésico: WGS84</b><br><br>
    Latitud: {miny:.6f} – {maxy:.6f}<br>
    Longitud: {minx:.6f} – {maxx:.6f}
  </div>

  <div style="border: 1px solid #ccc; border-radius: 8px; padding: 16px;">
    <b style="font-size: 15px;">Sistema Proyectado: UTM Zona 13N</b><br><br>
    X: {minx_u:.2f} – {maxx_u:.2f}<br>
    Y: {miny_u:.2f} – {maxy_u:.2f}
  </div>

</div>
"""
display(HTML(html))

AOI guardado en: /content/drive/MyDrive/ANP/AOI/aoi_QLA.geojson
CSV guardado en: /content/drive/MyDrive/ANP/AOI/QLA_coordenadas.csv


In [ ]:
# Cargar el área de estudio y reproyectar a WGS84
gdf = gpd.read_file(RUTA, layer=LAYER_NAME).to_crs(epsg=4326)

# Calcular bounding box
minx, miny, maxx, maxy = gdf.total_bounds
bbox_gdf = gpd.GeoDataFrame(
    geometry=[box(minx, miny, maxx, maxy)], crs=4326
)

# FIX: calcular centroide en CRS proyectado, luego volver a WGS84
centroid_proj = gdf.to_crs(epsg=3857).dissolve().centroid.iloc[0]

centroid = gpd.GeoSeries([centroid_proj], crs=3857).to_crs(epsg=4326).iloc[0]


#  Crear el mapa interactivo
mapa = geemap.Map()
mapa.set_center(centroid.x, centroid.y, ZOOM)

# Agregar área de estudio
mapa.add_gdf(gdf, layer_name=MAP_TITLE, style=STYLE_AREA)

# Agregar rectángulo rojo (bounding box)
mapa.add_gdf(bbox_gdf, layer_name='Bounding box', style=STYLE_BBOX)
print("Representación espacial")
# Mostrar el mapa
mapa